# DysCalc ML Component Pre-Processing, Training, and Evaluation

## 1.1 Dataset Labeling

In [ ]:
RAW_DATASET = RAW_DATASET_DIR / "FUNADB_rawdata_SUPPL.csv"
LABELED_DATASET = DATASET_DIR / "processed" / "FUNADB_labled.csv"
KEEP_COLUMNS = ["NC_t1", "DM_t1", "NS_t1", "ADD_t1", "SUB_t1", "CA_t1", "RMAT"]

# Read and Drop Unusable Columns
df_unl = pd.read_csv(RAW_DATASET, index_col=False)
df_unl = df_unl[KEEP_COLUMNS]
df_unl = df_unl.rename(columns=lambda c: c.removesuffix("_t1") if c != "RMAT" else c)   # remove the _t1 in the column names

# Label with 1 ("At-Risk") or 0 ("Typical") based on RMAT score
RMAT_scores = df_unl['RMAT'].to_numpy()

# Compute population mean and std for RMAT scores
RMAT_mean = np.mean(RMAT_scores)
RMAT_std = np.std(RMAT_scores)

# Normalize scores
RMAT_normalized = []
for score in RMAT_scores:
    RMAT_normalized.append((score - RMAT_mean) / RMAT_std)

# Labeling, with 35th percentile as threshold
RMAT_labels = []
threshold = np.percentile(RMAT_normalized, 35)
for normalized in RMAT_normalized:
    RMAT_labels.append(1 if normalized <= threshold else 0)

# Add the "Label" to the dataframe then drop RMAT since it will not be used 
df_unl['Label'] = np.array(RMAT_labels)
df_l = df_unl.drop(columns=['RMAT'])

df_l.to_csv(LABELED_DATASET, index=False)

df_l.head(10)

**Issue: RMAT `-99` sentinels labeled as At-Risk**

-   RMAT uses `-99` for missing.

-   Labeling runs before the sentinel cleanup, and the cleanup operates on `df_base` and skips the label column: it never touches the RMAT vector used to build the target.

-   RMAT `-99` enters labeling as a numeric −99 and always sits below any percentile cut.

-   42/358 children have `RMAT == -99`; **all 42 are labeled At-Risk**.

-   This is 30.4% of the At-Risk class (42/138): these positives encode "missing outcome," not "low numeracy."

-   **Consequence**: \~30% label noise in the minority class propagates to class balance, GAN training data, recall, and F2.

**Suggested Fix**

In [ ]:
df_unl = df_unl[df_unl['RMAT'] >= 0].reset_index(drop=True)
RMAT_scores = df_unl['RMAT'].to_numpy()

---
**Minor Issue: z-scoring is inert**

-   Labels are bit-identical whether thresholding z-scores or raw RMAT (percentiles are invariant under the monotone z-transform).

**Suggested Fix**

In [ ]:
threshold = np.percentile(RMAT_scores, 35)
RMAT_labels = (RMAT_scores <= threshold).astype(int)

## 1.2 Initial Exploratory Dataset Analysis

In [ ]:
df_raw = pd.read_csv(PROCESSED_DATASET_DIR / "FUNADB_labled.csv")

print("=== Dataset Info ===")
print(df_raw.info())

print("\n=== Dataset Head ===")
print(df_raw.head())

print("\n=== Summary Statistics (raw) ===")
print(df_raw.describe().T.round(2))

print("\n=== Missing / Sentinel Values ===")
DEFAULT_MISSING_VALUES = [-99]
sentinel_mask = df_raw.isin(DEFAULT_MISSING_VALUES)
print(f"Cells equal to {DEFAULT_MISSING_VALUES}:", sentinel_mask.sum().sum())
print(sentinel_mask.sum()[sentinel_mask.sum() > 0])

print("\n=== Label Distribution ===")
print(df_raw['Label'].value_counts())

In [ ]:
show_correlation_matrix(df_raw, 'raw', NUMBER_PROCESSING, ARITHMETIC_FLUENCY)

**Issue 1: correlation matrix computed with `-99` sentinels present**

-   `show_correlation_matrix` calls `df[number_processing + arithmetic_fluency].corr()` on `df_raw`, which still contains `-99`.

-   The sentinels act as extreme low outliers and attenuate every pairwise correlation, roughly halving them:

| pair   | as shown (with −99) | sentinel-excluded |
|--------|---------------------|-------------------|
| NC–NS  | −0.28               | −0.56             |
| ADD–CA | 0.50                | 0.77              |
| SUB–CA | 0.74                | 0.84              |
| NC–DM  | 0.64                | 0.72              |

-   **Consequence**: The notebook reads this matrix as confirming the two-factor structure (Number Processing against Arithmetic Fluency) and the expected cross-block negative correlations. Any downstream reasoning about collinearity or feature redundancy from this matrix is based on attenuated numbers.

**Suggested Fix**

In [ ]:
df[cols].replace(-99, np.nan).corr()

---

In [ ]:
def show_feature_class_outliers(df, df_type: str, feature_columns: list):
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    axes = axes.flatten()

    for i, col in enumerate(feature_columns):
        ax = axes[i]
        data_to_plot = [df[df['Label'] == lbl][col].dropna() for lbl in [0, 1]]
        bp = ax.boxplot(data_to_plot, patch_artist=True, notch=False,
                        medianprops=dict(color='black', linewidth=2))
        for patch, color in zip(bp['boxes'], [TYPICAL_COLOR, AT_RIST_COLOR]):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_xticklabels(['Class 0', 'Class 1'])
        ax.set_title(col)
        ax.set_ylabel('Value')

    plt.tight_layout()
    plt.savefig(FIGURES_OUTPUTS / f'boxplots_{df_type}.png')
    plt.show()

In [ ]:
show_feature_distribution_plot(df_raw, 'raw', FEATURE_COLUMNS, DEFAULT_MISSING_VALUES)

**Issue 2: Boxplots plot sentinels as outliers**

-   `.dropna()` does not catch `-99` (it is a value, not NaN), so all 59 sentinels render as extreme-low outliers.

-   The next cell reasons about per-class outliers and IQR clipping from the contaminated plots.

## 1.3 Dataset Cleanup

In [ ]:
train_medians = df_train_flagged[FEATURE_COLUMNS].median()
train_medians = train_medians.fillna(0)

df_train_imputed = df_train_flagged.copy()
df_val_imputed   = df_val_flagged.copy()
df_test_imputed  = df_test_flagged.copy()

for df_tmp in (df_train_imputed, df_val_imputed, df_test_imputed):
    df_tmp[FEATURE_COLUMNS] = df_tmp[FEATURE_COLUMNS].fillna(train_medians)

In [ ]:
def compute_iqr_bounds(df: pd.DataFrame, feature_columns: list, whisker: float = 1.5) -> dict:
    bounds = {}
    for col in feature_columns:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - whisker * iqr
        upper = q3 + whisker * iqr
        bounds[col] = (lower, upper)
    return bounds

def apply_clip(df: pd.DataFrame, bounds: dict) -> pd.DataFrame:
    df_out = df.copy()
    for col, (lo, hi) in bounds.items():
        df_out[col] = df_out[col].clip(lo, hi)
    return df_out

clip_bounds = compute_iqr_bounds(df_train_imputed, FEATURE_COLUMNS)

df_train_clean = apply_clip(df_train_imputed, clip_bounds).reset_index(drop=True)
df_val_clean   = apply_clip(df_val_imputed, clip_bounds).reset_index(drop=True)
df_test_clean  = apply_clip(df_test_imputed, clip_bounds).reset_index(drop=True)

# Full cleaned dataset for descriptive analysis (train-fitted preprocessing)
df_clean = pd.concat([df_train_clean, df_val_clean, df_test_clean], ignore_index=True)

print('Training-derived clipping bounds applied:')
for col, (lo, hi) in clip_bounds.items():
    print(f'  {col}: [{lo:.2f}, {hi:.2f}]')

**Issue: Imputation precedes clip-bound computation**

-   Imputed values sit exactly at the median, pulling Q1 and Q3 inward and slightly tightening bounds.

-   Measured effect is small but nonzero and largest where missingness is highest:

| feature | bounds on imputed | bounds pre-impute (NaN-excluded) |
|---------|-------------------|----------------------------------|
| NC      | (157.6, 1739.5)   | (150.4, 1750.2)                  |
| DM      | (86.3, 4025.0)    | (77.0, 4037.0)                   |
| SUB     | (0.5, 68.5)       | (−3.5, 72.5)                     |

-   Quantiles should be computed on observed values, not on values pinned to the median.

**Suggested Fix**

In [ ]:
compute_iqr_bounds(df_train_flagged, FEATURE_COLUMNS)